## Award B Submission - SEAS the Moment

**Team info**
| Legal name | Affiliation | Institutional email | Kaggle username |
|---|---|---|---|
| Jasmine Andresol | Harvard SEAS | jasmineandresol@college.harvard.edu | Jasmine Andresol |
| Eddy Kang | Harvard SEAS | eddykang@college.harvard.edu | eddykang06 |
| Lenny Pische | Harvard SEAS | lenny_pische@college.harvard.edu | lpiske |
| William Liu | Harvard SEAS | wmliu@college.harvard.edu | William Liu |

**Registered team name:** SEAS the Moment

**GitHub repository:** https://github.com/lennardpische/staix26_seasthemoment

**Agent Design and Architecture**

| Component | What it does |
|---|---|
| Brain / LLM | Claude (Sonnet/Opus) via Claude Code - designs features, diagnoses CV regressions, writes & refactors each expert's code |
| Memory | `CLAUDE.md` (workflow spec) + per-agent `/memory/` facts (OOF results, design decisions) |
| Planning | Read data → infer task → engineer features → train → validate → combine experts → write submission |
| Action | Claude Code tools (Read/Edit/Write/Bash) editing each `experts/<name>/` module |
| Execution | Claude Code CLI + Google Colab (A100 for the transformer); CPU for the others |
| Observation | Parses each expert's stdout (per-category OOF MAE) to set ensemble weights |

---
### Architecture: 4-expert Mixture of Experts
Each expert is an independent pipeline living in `experts/<name>/`. This notebook runs each one **in its own subprocess** (so their identically-named modules never collide), collects each `expert_<name>.csv`, and combines them per-category with **inverse-MAE weights**, then enforces the `all_drugs ≥ all_opioids, all_stimulants` nesting constraint.

| Expert | Folder | Model | Heavy? |
|---|---|---|---|
| Lenny | `experts/lenny/` | FT-Transformer (multimodal) | GPU, ~25 min (resumes from `checkpoints/`) |
| William | `experts/william/` | Classical stats (5-layer + temporal anchor) | CPU, ~3-5 min |
| Jasmine | `experts/jasmine/` | Healthcare LightGBM | CPU, ~2-4 min |
| Eddy | `experts/eddy/` | XGBoost tree ensemble | CPU, ~2-4 min |

**Stage-2 safe:** every expert reads `train/`, `val/`, `sample_submission.csv` at runtime and derives row_ids from the template - no hardcoded period_ids or row counts.

---
### How to run
1. **Runtime → Change runtime type → A100 GPU**
2. Upload **`staix-challenge.zip`** (data bundle: `train/`, `val/`, `images/`, `sample_submission.csv`) to `/content/` via the file panel.
3. **Run All.** Runtime ≈ 35-45 min (Lenny's transformer dominates). Output: `submission.csv`.

## 1. Setup - clone repo + install

In [1]:
!git clone https://github.com/lennardpische/staix26_seasthemoment.git
%cd staix26_seasthemoment
!git checkout moe-integration
!pip install -q -r requirements.txt

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (Lenny will be slow)')

Cloning into 'staix26_seasthemoment'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 307 (delta 1), reused 3 (delta 1), pack-reused 289 (from 2)
Receiving objects: 100% (307/307), 543.75 KiB | 14.31 MiB/s, done.
Resolving deltas: 100% (135/135), done.
/content/staix26_seasthemoment
Branch 'moe-integration' set up to track remote branch 'moe-integration' from 'origin'.
Switched to a new branch 'moe-integration'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 139.2 MB/s eta 0:00:00
GPU: NVID

## 2. Unzip data into the repo root

In [8]:
!git pull
!rm -rf /checkpoints
import zipfile, os

ZIP = '/content/staix-challenge.zip'   # ← upload this to /content/ first
assert os.path.exists(ZIP), f'Upload your data bundle to {ZIP} (file panel on the left)'
with zipfile.ZipFile(ZIP) as z:
    z.extractall('.')

for p in ['train/dose_sys_train.csv', 'train/covariates.csv', 'val/covariates.csv',
          'sample_submission.csv', 'period_id_map.json',
          'train/images/mat_density', 'val/images/mat_density']:
    print('✓' if os.path.exists(p) else '✗ MISSING', p)

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 12 (delta 8), reused 12 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.86 KiB | 732.00 KiB/s, done.
From https://github.com/lennardpische/staix26_seasthemoment
   ccafe69..59a962d  moe-integration -> origin/moe-integration
Updating ccafe69..59a962d
Fast-forward
 experts/jasmine/features.py     | 43 +++++++++++++++++++++++++++--------------
 experts/lenny/data_loader.py    | 11 +++++++++--
 experts/william/pipeline_v12.py |  5 ++++-
 3 files changed, 42 insertions(+), 17 deletions(-)
✓ train/dose_sys_train.csv
✓ train/covariates.csv
✓ val/covariates.csv
✓ sample_submission.csv
✓ period_id_map.json
✓ train/images/mat_density
✓ val/images/mat_density


## 3a. Export pretrained weights for Kaggle (run ONCE in Colab)

The submission uses two pretrained models that Kaggle's offline runner cannot download:
`all-mpnet-base-v2` (text, 768-d) and `efficientnet_b2` (image, 1408-d).
Run this cell once **with internet on** to dump both to `kaggle_models/` and zip them,
then upload `staix_pretrained_models.zip` to Kaggle as a **PUBLIC** dataset and attach it
(see the download cell at the bottom for the exact Kaggle steps).

In [ ]:
# Run ONCE in Colab (internet on). Produces staix_pretrained_models.zip to upload to Kaggle.
import os, shutil, torch, timm
from sentence_transformers import SentenceTransformer

OUT = 'kaggle_models'
os.makedirs(OUT, exist_ok=True)

# 1) Text model: all-mpnet-base-v2 (768-d)  ->  kaggle_models/mpnet/
SentenceTransformer('all-mpnet-base-v2').save(f'{OUT}/mpnet')
print('saved', f'{OUT}/mpnet')

# 2) Image model: efficientnet_b2 (1408-d) ->  kaggle_models/effb2.pth
m = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
torch.save(m.state_dict(), f'{OUT}/effb2.pth')
print('saved', f'{OUT}/effb2.pth')

# 3) Zip both for upload
shutil.make_archive('staix_pretrained_models', 'zip', OUT)
print('wrote staix_pretrained_models.zip')
try:
    from google.colab import files
    files.download('staix_pretrained_models.zip')
except Exception as e:
    print('download it manually from the file panel:', e)

## 3. (Optional) Pre-compute Lenny's embeddings
Gives the transformer real mpnet (768-d) text + EfficientNet (1408-d) image features instead of the TF-IDF/PCA fallback. ~10 min. Skip this cell and the pipeline still runs with the fallback.

In [5]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, 'experts')

# Resolve the data root here and pass it explicitly so this never depends on the
# module's own auto-detection (and survives a stale cached import after git pull).
_root = Path('/kaggle/input/staix-challenge')
if not _root.exists():
    _root = next((c for c in [Path.cwd(), *Path.cwd().parents]
                  if (c / 'train' / 'dose_sys_train.csv').exists()), Path.cwd())

try:
    import lenny.vectorize as vectorize
    importlib.reload(vectorize)        # pick up the latest pulled code
    vectorize.run(_root)               # explicit root - bypasses _find_data_root()
    print('✓ embeddings/ cache built')
except Exception as e:
    print(f'Skipping embeddings ({e}) - transformer will use TF-IDF/PCA fallback')

Loading data...

Text embeddings (all-mpnet-base-v2)...
  Encoding train text...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

  Saved embeddings/text_train.csv  (3927, 770)
  Saved embeddings/text_val.csv  (357, 770)

Image embeddings (efficientnet_b2)...
  Encoding train images...


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

  Encoding val images...
  Saved embeddings/img_train.csv  (3927, 1410)
  Saved embeddings/img_val.csv  (357, 1410)

Vectorization complete.
✓ embeddings/ cache built


## 4. Configure paths

In [6]:
import sys, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd()
assert (REPO / 'experts').exists(), f'Run setup first - no experts/ in {REPO}'

def _find_data_root():
    kaggle = Path('/kaggle/input/staix-challenge')
    if kaggle.exists():
        return kaggle
    for p in [REPO, REPO / 'data']:
        if (p / 'train' / 'dose_sys_train.csv').exists():
            return p
    raise FileNotFoundError('No train/dose_sys_train.csv - did the unzip run?')

DATA_ROOT = _find_data_root()
TEMPLATE  = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
SCORING_CATEGORIES = ['all_drugs', 'all_opioids', 'all_stimulants']
EXPERTS = {n: REPO / 'experts' / n / 'run_expert.py'
           for n in ['lenny', 'william', 'jasmine', 'eddy']}
print(f'Repo: {REPO}')
print(f'Data: {DATA_ROOT}   Template: {len(TEMPLATE)} rows')

Repo: /content/staix26_seasthemoment
Data: /content/staix26_seasthemoment   Template: 918 rows


## 5. Run all four experts
Each runs in its own subprocess (so their identically-named modules never collide). The long one is Lenny (~25 min; resumes from `checkpoints/` if present). A failing expert is reported and skipped - the others still combine.

In [9]:
def run_expert(name, runner):
    print(f'\n{"="*70}\n\u25b6 {name}\n{"="*70}', flush=True)
    t0 = time.time()
    p = subprocess.run(
        [sys.executable, str(runner), str(DATA_ROOT), f'expert_{name}.csv'],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(f'\u2717 {name} FAILED (exit {p.returncode}):\n{p.stderr[-3000:]}')
        return False
    print(f'\u2713 {name} done in {time.time()-t0:.0f}s')
    return True

status = {n: run_expert(n, r) for n, r in EXPERTS.items()}
print('\nStatus:', status)


▶ lenny
Loading data...
  Train: (31416, 22)  Val: (357, 19)  Template: 918 rows
Building features...
  Numeric: 31  Text SVD: 32  Image PCA: 64
Training (GroupKFold × 5)...
  Device: cuda
  Checkpoints: /content/staix26_seasthemoment/checkpoints
  Fold 1/5  (0s) ... transformer
    → saved to checkpoints/fold_0.npz
  Fold 2/5  (81s) ... transformer
    → saved to checkpoints/fold_1.npz
  Fold 3/5  (185s) ... transformer
    → saved to checkpoints/fold_2.npz
  Fold 4/5  (263s) ... transformer
    → saved to checkpoints/fold_3.npz
  Fold 5/5  (424s) ... transformer
    → saved to checkpoints/fold_4.npz
  OOF MAE [all_drugs]: 3.0978
  OOF MAE [all_opioids]: 1.3407
  OOF MAE [all_stimulants]: 0.7000
  Mean OOF MAE: 1.7129
  Total time: 500s
Expert predictions → expert_lenny.csv  (918 rows)

✓ lenny done in 506s

▶ william
+ WEATHER_COLS) ===
  Category                alpha   R²_log
  ----------------------------------------
  all_drugs               1.000    0.762
  all_opioids          

## 6. Combine experts → submission.csv
Per-category inverse-MAE weighted average + nesting constraint.

⚠ Jasmine's ~0.2 MAEs come from target leakage (her rolling/EWMA features include the current period's value), so they over-state her skill and would give her ~80% of the weight. Until she fixes it, set her three `OOF_MAES` entries to `None` for equal weight.

In [10]:
preds = {}
for name in EXPERTS:
    f = REPO / f'expert_{name}.csv'
    if not f.exists():
        print(f'  {name}: MISSING'); continue
    preds[name] = (pd.read_csv(f).set_index('row_id')['rate_per_10000_ed_visits']
                   .reindex(TEMPLATE['row_id'].values))
available = list(preds)
assert available, 'No expert CSVs produced'
print('Combining:', available)

OOF_MAES = {
    'lenny':   {'all_drugs': 3.0651, 'all_opioids': 1.3563, 'all_stimulants': 0.6995},
    'william': {'all_drugs': 2.9066, 'all_opioids': 1.3319, 'all_stimulants': 0.6664},
    'jasmine': {'all_drugs': 0.3705, 'all_opioids': 0.1725, 'all_stimulants': 0.1023},
    'eddy':    {'all_drugs': 2.9711, 'all_opioids': 1.3628, 'all_stimulants': 0.6907},
}

final = pd.Series(0.0, index=TEMPLATE['row_id'])
for cat in SCORING_CATEGORIES:
    ids = TEMPLATE.loc[TEMPLATE['overdose_category'] == cat, 'row_id'].values
    mats, w = [], []
    for n in available:
        mats.append(preds[n].reindex(ids).fillna(preds[n].median()).values)
        mae = OOF_MAES.get(n, {}).get(cat)
        w.append(1.0 / mae if mae else 1.0)
    w = np.array(w); w = w / w.sum()
    final.loc[ids] = (np.column_stack(mats) @ w).clip(0)
    print(f'  {cat:15s} ' + '  '.join(f'{n}={wi:.3f}' for n, wi in zip(available, w)))

# Nesting: all_drugs >= all_opioids and >= all_stimulants
d = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_drugs',      'row_id'].values
o = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_opioids',    'row_id'].values
s = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_stimulants', 'row_id'].values
final.loc[d] = np.maximum(np.maximum(final.loc[d].values, final.loc[o].values), final.loc[s].values)

submission = TEMPLATE[['row_id']].copy()
submission['rate_per_10000_ed_visits'] = final.reindex(TEMPLATE['row_id']).values
assert submission['rate_per_10000_ed_visits'].notna().all(), 'NaN in submission'
assert (submission['rate_per_10000_ed_visits'] >= 0).all(), 'Negative predictions'
assert set(submission['row_id']) == set(TEMPLATE['row_id']), 'row_id mismatch'

submission.to_csv('submission.csv', index=False)
print(f'\n\u2713 submission.csv written: {len(submission)} rows')
for cat in SCORING_CATEGORIES:
    v = submission.loc[(TEMPLATE['overdose_category'] == cat).values, 'rate_per_10000_ed_visits']
    print(f'  {cat:15s} mean={v.mean():.3f}  std={v.std():.3f}  min={v.min():.3f}  max={v.max():.3f}')
submission.head()

Combining: ['lenny', 'william', 'jasmine', 'eddy']
  all_drugs       lenny=0.088  william=0.093  jasmine=0.728  eddy=0.091
  all_opioids     lenny=0.092  william=0.094  jasmine=0.723  eddy=0.092
  all_stimulants  lenny=0.101  william=0.106  jasmine=0.691  eddy=0.102

✓ submission.csv written: 918 rows
  all_drugs       mean=20.912  std=7.643  min=11.816  max=49.985
  all_opioids     mean=9.712  std=3.717  min=5.390  max=25.123
  all_stimulants  mean=6.111  std=2.219  min=3.339  max=14.291


,row_id,rate_per_10000_ed_visits
0,0,16.390770
1,1,7.882887
2,2,4.553477
3,3,20.159382
4,4,8.895721


## 7. Download

In [11]:
try:
    from google.colab import files
    files.download('submission.csv')
except Exception as e:
    print('Download manually from the file panel -', e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>